# 03 — Delay Prediction (XGBoost Classifier)

**Goal:** Train an XGBoost Classifier to predict `IsDelayed` (1 if project duration exceeds the historical median, 0 otherwise).

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_parquet('../model_artifacts/processed_data.parquet')
config = joblib.load('../model_artifacts/feature_config.pkl')

delay_features = config['delay_features']
delay_target = config['delay_target']

print(f'Features ({len(delay_features)}): {delay_features}')
print(f'Target: {delay_target}')

## 2. Train/Test Split

In [ ]:
X = df[delay_features]
y = df[delay_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Class balance (Train):\n{y_train.value_counts(normalize=True)}')


## 3. Train XGBoost Classifier

In [ ]:
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)

clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=10
)

## 4. Evaluation

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred))

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 5. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': delay_features,
    'Importance': clf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance (Delay Prediction)', fontsize=14, fontweight='bold')
plt.show()

## 6. Export Model Artifact

In [ ]:
model_path = '../model_artifacts/xgb_delay_model.pkl'
joblib.dump(clf, model_path)
print(f'Saved delay model to {model_path}')